ref: https://github.com/sweenip/jupyter#selecting-a-python-interpreter-for-jupyter-notebooks-in-vs-code

# Setup in wsl bash
```bash
cd sweeni
uv venv
source .venv/bin/activate
uv pip install -r analyze_requirements.txt
```

```bash
# Resolved 37 packages in 21ms
# Installed 37 packages in 86ms
#  + asttokens==3.0.0
#  + comm==0.2.2
#  + debugpy==1.8.14
#  + decorator==5.2.1
#  + et-xmlfile==2.0.0
#  + executing==2.2.0
#  + ipykernel==6.29.5
#  + ipython==9.4.0
#  + ipython-pygments-lexers==1.1.1
#  + jedi==0.19.2
#  + jinja2==3.1.6
#  + jupyter-client==8.6.3
#  + jupyter-core==5.8.1
#  + markupsafe==3.0.2
#  + matplotlib-inline==0.1.7
#  + nest-asyncio==1.6.0
#  + numpy==2.3.1
#  + openpyxl==3.1.5
#  + packaging==25.0
#  + pandas==2.3.1
#  + parso==0.8.4
#  + pexpect==4.9.0
#  + platformdirs==4.3.8
#  + prompt-toolkit==3.0.51
#  + psutil==7.0.0
#  + ptyprocess==0.7.0
#  + pure-eval==0.2.3
#  + pygments==2.19.2
#  + python-dateutil==2.9.0.post0
#  + pytz==2025.2
#  + pyzmq==27.0.0
#  + six==1.17.0
#  + stack-data==0.6.3
#  + tornado==6.5.1
#  + traitlets==5.14.3
#  + tzdata==2025.2
#  + wcwidth==0.2.13
```

# Select interpreter

https://github.com/sweenip/jupyter#selecting-a-python-interpreter-for-jupyter-notebooks-in-vs-code

Note the interpreter name on top right

![alt text](assets/test_interpreter.png)

If right interpreter does not exist then create one
1. VSCode was started in, say, `~/code/`
1. `Ctrl+Shift+P` : Python Create Environment
1. `select environment type` : venv
1. `select a python installation to create the virtial environment` : select `/usr/bin/python3` [Global]
1. `select dependencies to install` : requirements were in `OneDriveExplorer/sweeni/analyze_requirements.txt`, so slected that
1. VSCode created an env in `~/code/.venv` and showed message : `The following environment is selected: ~/code/.venv/bin/python`

`~/code/.venv/pyvenv.cfg` contains:
```
home = /usr/bin
include-system-site-packages = false
version = 3.12.3
executable = /usr/bin/python3.12
command = /usr/bin/python3 -m venv --without-pip /home/sweeni/code/.venv
```
Appended line
```
prompt = OneDriveExplorer_sweeni
```

In [ ]:
# Test
!pip show numpy

Extract data from OneDrive Database on WINDOWS


```bash
cd /c/D/code/OneDriveExplorer
source .venv_win/Scripts/activate
ls /c/D/code/OneDriveExplorer/tmp/ztsdaf1471/OneDrive_250714
# SafeDelete.db  SettingsDatabase.db  SyncEngineDatabase.db
python ./OneDriveExplorer/OneDriveExplorer.py -s //c:/D/code/OneDriveExplorer/tmp/ztsdaf1471/OneDrive_250714 --csv tmp
```
Copy `/c/D/code/OneDriveExplorer/tmp/SQLite_DB_OneDrive.csv` to linux under `/home/sweeni/code/OneDriveExplorer/data/ztsdaf1471/OneDrive_250714`

In [ ]:
import pandas as pd
import pickle

In [ ]:
# `file` was created on WINDOWS using https://github.com/sweenip/OneDriveExplorer/blob/dev/sweeni/ztsdaf1471.md
folder = '/home/sweeni/code/OneDriveExplorer/data/ztsdaf1471/OneDrive_250714'
file="SQLite_DB_OneDrive.csv"

In [ ]:
columns=["Type","scopeID","siteID","webID","listID","tenantID","webURL","remotePath","spoPermissions","shortcutVolumeID","shortcutItemIndex","libraryType","parentResourceID","resourceID","eTag","Name","fileStatus","lastKnownPinState","volumeID","itemIndex","diskLastAccessTime","diskCreationTime","lastChange","size","localHashDigest","localHashAlgorithm","sharedItem","firstHydrationTime","lastHydrationTime","hydrationCount","lastHydrationType","Media","parentScopeID","folderStatus","folderColor","MountPoint","Path","fileName","graphMetadataJSON","spoCompositeID","createdBy","modifiedBy","filePolicies","fileExtension","lastWriteCount"]

required_fields = ['Type', "parentResourceID","resourceID", "Name", 'fileStatus', "volumeID", "itemIndex", 'diskCreationTime', 'lastChange', 'size', "parentScopeID",'folderStatus', 'Path', 'fileName', 'lastWriteCount']

In [ ]:
df = pd.read_csv(folder + "/" + file, skipinitialspace=True, usecols=required_fields,
                 converters={'diskCreationTime': pd.to_datetime,
                             'lastChange': pd.to_datetime,
                            "size": lambda x: x.replace(" KB", "").replace(",", ""),
                             }, 
                 dtype={'Type' : 'category',
                        'fileStatus' : 'category',
                        'folderStatus' : 'category',
                        "volumeID" : 'category',
                        "parentScopeID" : 'category',
                        "lastWriteCount" : pd.Int64Dtype,
                        # used  Nullable Integer Dtypes instaed of np.int64 to avoid error 
                        # "ValueError: Integer column has NA values in column NN"
                        }
                )

In [ ]:
df['size'] = pd.to_numeric(df['size'], errors='coerce')  # Convert size to Int64 dtype
#Reorder

ordered_fields = ['Path', 'fileName', "Name", 'Type', 'size', 'lastChange',  'diskCreationTime', 'lastWriteCount',"parentResourceID","resourceID", 'fileStatus', 'folderStatus', "volumeID", "itemIndex", "parentScopeID"]
df = df[ordered_fields]
# df.head()

In [ ]:
with open(folder + "/" + 'out.pickle','wb') as f:
    pickle.dump(df,f)

In [ ]:
with open(folder + "/" + 'out.pickle','rb') as f:
    df=pickle.load(f)

In [ ]:
# df['parentScopeID'].unique()  # Check unique values in parentScopeID
# df9e9e556244467b9dabdc62d965839f may be sweeni@hotmail.com


In [ ]:
# df[(df['fileName']!=df['Name'])&(df['Type']!="Folder")].head()

In [ ]:
# df[df['Type']=="Scope"].head()

In [ ]:
df['Type'].unique() 
# ['File', 'Folder']
# Categories (3, object): ['File', 'Folder', 'Scope']

In [ ]:
# Remove the Scope record
df=df[df['Type']!="Scope"]

# File or Folder
# Remove prefix
df['Path']=df['Path'].str.replace("df9e9e556244467b9dabdc62d965839f", "")
#df.head()

In [ ]:
df["FullName"] = df["Path"] + "\\" + df["Name"]
ordered_fields = ["FullName", 'Type', 'size', 'lastChange',  'diskCreationTime', 'lastWriteCount',"parentResourceID","resourceID", 'fileStatus', 'folderStatus', "volumeID", "itemIndex", "parentScopeID"] # 'Path', 'fileName', "Name",
df = df[ordered_fields]

In [ ]:
status_map = {
    1: "ADDED",
    2: "MODIFIED",
    6: "DOWNLOADED",
    7: "UPLOADING",
    8: "UPLOADED",
    9: "CONFLICT",
   11: "ERROR",
}

df['fileStatusName'] = status_map.get(df['fileStatus'], f"UNKNOWN({df['fileStatus']})")

In [ ]:
phrases = ['\\\\ztsdaf1471\\', '\\\\_wip\\', '\\\\D\\', '\\\\data\\', '\\\\temp\\', '\\\\Attachments\\','\\\\ToRead\\', '\\\\shared\\']
filtered_df = df[~df["FullName"].str.startswith(tuple(phrases))]

Perhaps export this to `./treeviw` 

Can also look into treesize or whereisit to visualize with analysis

# Directly connecting to SQLite database 

In [ ]:
# import os
# import sqlite3

# # Path to your SQLite OneDrive DB
# db_path = os.path.expanduser("~/code/OneDriveExplorer/data/ztsdaf1471/OneDrive_250714/SyncEngineDatabase.db")

In [ ]:
# conn = sqlite3.connect(db_path)
# cursor = conn.cursor()

# # Fetch Personal Vault entries
# cursor.execute("""
#   SELECT ItemID, Path, FileStatus, LastSyncTime
#   FROM ItemProperties
#   WHERE Path LIKE '%Personal Vault%'
#   ORDER BY LastSyncTime DESC
#     LIMIT 10
# """)

# for item_id, path, status, last_sync in cursor.fetchall():
#     print(f"{item_id}\t{status}\t{last_sync}\t{path}")

# cursor.close()
# conn.close()